<a href="https://colab.research.google.com/github/apopuri584/wriggling_bad2/blob/main/PF_sam_single_video_pipeline_optimized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAM-based worm segmentation — single video (Colab upload)

Author: Anoushka Popuri

Modifications: Patrick Flynn

**This version processes ONE video that you upload directly to this Colab session** (no Google Drive needed). Just run the cells in order; you'll be prompted to pick a file when you get to the upload cell.


In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
import os, csv, glob, sys, random
import cv2, torch, numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_fill_holes
from skimage.morphology import skeletonize
from skimage.measure import regionprops
%matplotlib inline

try:
    from segment_anything import sam_model_registry, SamPredictor
except ImportError:
    !pip install git+https://github.com/facebookresearch/segment-anything.git -q
    from segment_anything import sam_model_registry, SamPredictor

if not os.path.exists('sam_vit_b_01ec64.pth'):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    print('SAM checkpoint downloaded.')
else:
    print('SAM checkpoint already present.')

from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))


  Preparing metadata (setup.py) ... done
SAM checkpoint downloaded.


In [ ]:
# ── Cell 2: CONFIGURATION ──────────────────────────────────────────────────

# Where to save the segmented output (created if it doesn't exist).
# Lives in the Colab session storage (/content), not Google Drive, since
# we're only processing one uploaded video.
OUTPUT_ROOT = '/content/output'

# ── Detection & analysis parameters (tune per your original notebook) ─────────
N_BG_FRAMES        = 30
BG_THRESH          = None   # None = Otsu auto-threshold
MIN_WORM_AREA      = 50
MAX_WORM_AREA      = 50_000
MIN_ASPECT_RATIO   = 1.2
MIN_SOLIDITY       = 0.15
BOX_PADDING        = 10
TRACKING_IOU_THRESH = 0.1

WORM_COLORS = [
    (0,   255, 0),    # green
    (255, 100, 0),    # orange
    (0,   180, 255),  # cyan
    (255, 0,   200),  # magenta
    (255, 255, 0),    # yellow
    (150, 0,   255),  # purple
    (0,   255, 180),  # teal
    (255, 50,  50),   # red
    (50,  50,  255),  # blue
    (255, 200, 0),    # gold
]

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f'Output : {OUTPUT_ROOT}')


Output : /content/output


In [ ]:
# ── Cell 3: Upload your video ─────────────────────────────────────────────
# Click 'Choose Files' and select ONE video from your computer.
# It will be saved into this Colab session at /content/<filename>.

from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, 'Please upload exactly one video file.'

video_name = next(iter(uploaded.keys()))
video_path = os.path.join('/content', video_name)
print(f'Video ready at: {video_path}')


Saving 20mM caffeine 3minutes.MOV to 20mM caffeine 3minutes.MOV
Video ready at: /content/20mM caffeine 3minutes.MOV


In [ ]:
# ── Cell 3: Load SAM model ────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('⚠ WARNING: No GPU detected! SAM will be extremely slow on CPU.')
    print('  In Colab: Runtime → Change runtime type → Hardware accelerator → GPU (T4), then re-run all cells.')

sam = sam_model_registry['vit_b'](checkpoint='sam_vit_b_01ec64.pth')
sam.to(device)
predictor = SamPredictor(sam)
print('SAM model loaded.')


Using device: cuda
SAM model loaded.


In [ ]:
# ── Cell 4: Helper functions (identical to your original notebook, + speed fixes) ─

def build_background(video_path, n_frames=N_BG_FRAMES):
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step  = max(1, total // n_frames)
    frames = []
    for i in range(0, min(total, n_frames * step), step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    cap.release()
    if not frames:
        raise RuntimeError('Could not read any frames for background estimation.')
    return np.median(np.stack(frames, axis=0), axis=0).astype(np.uint8)


def detect_worm_boxes(gray_frame, background,
                      bg_thresh=BG_THRESH,
                      min_area=MIN_WORM_AREA, max_area=MAX_WORM_AREA,
                      min_aspect=MIN_ASPECT_RATIO, min_solidity=MIN_SOLIDITY,
                      padding=BOX_PADDING):
    h, w = gray_frame.shape
    bright = cv2.subtract(gray_frame, background)
    if bg_thresh is None:
        _, fg_mask = cv2.threshold(bright, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        _, fg_mask = cv2.threshold(bright, bg_thresh, 255, cv2.THRESH_BINARY)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN,  kernel, iterations=1)
    num_labels, label_img, stats, _ = cv2.connectedComponentsWithStats(fg_mask, connectivity=8)
    boxes = []
    for lbl in range(1, num_labels):
        area = stats[lbl, cv2.CC_STAT_AREA]
        if not (min_area <= area <= max_area):
            continue
        blob  = (label_img == lbl).astype(np.uint8)
        props = regionprops(blob)[0]
        aspect = (props.major_axis_length / props.minor_axis_length
                  if props.minor_axis_length > 0 else 0)
        if aspect < min_aspect or props.solidity < min_solidity:
            continue
        x  = stats[lbl, cv2.CC_STAT_LEFT]
        y  = stats[lbl, cv2.CC_STAT_TOP]
        bw = stats[lbl, cv2.CC_STAT_WIDTH]
        bh = stats[lbl, cv2.CC_STAT_HEIGHT]
        boxes.append(np.array([max(0, x-padding), max(0, y-padding),
                                 min(w-1, x+bw+padding), min(h-1, y+bh+padding)]))
    return boxes, fg_mask

def segment_boxes_batched(predictor, image_rgb, boxes):

    predictor.set_image(image_rgb)
    if len(boxes) == 0:
        return []
    boxes_np = np.stack(boxes).astype(np.float32)
    boxes_t  = torch.as_tensor(boxes_np, device=predictor.device)
    transformed_boxes = predictor.transform.apply_boxes_torch(boxes_t, image_rgb.shape[:2])
    with torch.no_grad():
        masks, _, _ = predictor.predict_torch(
            point_coords=None,
            point_labels=None,
            boxes=transformed_boxes,
            multimask_output=False,
        )
    masks = masks.squeeze(1).to(torch.uint8).cpu().numpy()  # (N, H, W)
    return [masks[i] for i in range(masks.shape[0])]


.
def skeletonize_mask(mask):
    ys, xs = np.where(mask == 1)
    skeleton = np.zeros(mask.shape, dtype=bool)
    if len(xs) == 0:
        return skeleton, 0, [], 0.0

    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    crop = mask[y0:y1, x0:x1].astype(bool)
    skel_crop = skeletonize(crop)
    skeleton[y0:y1, x0:x1] = skel_crop

    length_px = int(skel_crop.sum())
    endpoints = []
    skel_pts_crop = np.argwhere(skel_crop)
    if len(skel_pts_crop) > 2:
        for (r, c) in skel_pts_crop:
            patch = skel_crop[max(0,r-1):r+2, max(0,c-1):c+2]
            if patch.sum() == 2:
                endpoints.append((r + y0, c + x0))

    curvature = 0.0
    if len(skel_pts_crop) >= 3:
        pca_vec = np.cov(skel_pts_crop.T)
        _, evecs = np.linalg.eigh(pca_vec)
        proj    = skel_pts_crop @ evecs[:, -1]
        ordered = skel_pts_crop[np.argsort(proj)]
        sampled = ordered[::5]
        if len(sampled) >= 3:
            angles = []
            for i in range(1, len(sampled) - 1):
                v1 = sampled[i] - sampled[i-1]
                v2 = sampled[i+1] - sampled[i]
                cos_a = np.dot(v1, v2) / (np.linalg.norm(v1)*np.linalg.norm(v2) + 1e-8)
                angles.append(np.arccos(np.clip(cos_a, -1, 1)))
            curvature = float(np.mean(angles)) if angles else 0.0
    return skeleton, length_px, endpoints, curvature


def mask_iou(m1, m2):
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    return inter / union if union > 0 else 0.0


def match_masks_to_tracks(prev_masks_dict, new_masks,
                           iou_thresh=TRACKING_IOU_THRESH,
                           next_id_ref=[0]):
    updated, used_new = {}, set()
    scores = [(mask_iou(pm, nm), tid, j)
               for tid, pm in prev_masks_dict.items()
               for j, nm in enumerate(new_masks)]
    scores.sort(reverse=True)
    matched_tids = set()
    for iou, tid, j in scores:
        if iou < iou_thresh: break
        if tid in matched_tids or j in used_new: continue
        updated[tid] = new_masks[j]; matched_tids.add(tid); used_new.add(j)
    for j, nm in enumerate(new_masks):
        if j not in used_new:
            updated[next_id_ref[0]] = nm; next_id_ref[0] += 1
    return updated


def contour_skeleton_and_overlap(sam_mask, sam_skeleton):
    ys, xs = np.where(sam_mask == 1)
    contour_skel = np.zeros(sam_mask.shape, dtype=bool)
    if len(xs) == 0:
        return contour_skel, 0, 0.0, 0.0

    pad = 2  # room for the dilate/erode below
    y0 = max(0, ys.min() - pad); y1 = min(sam_mask.shape[0], ys.max() + 1 + pad)
    x0 = max(0, xs.min() - pad); x1 = min(sam_mask.shape[1], xs.max() + 1 + pad)

    mask_crop = sam_mask[y0:y1, x0:x1]
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    dilated = cv2.dilate(mask_crop, kernel, iterations=1)
    eroded  = cv2.erode(mask_crop,  kernel, iterations=1)
    filled  = binary_fill_holes(cv2.subtract(dilated, eroded).astype(bool)).astype(np.uint8)
    contour_skel_crop = skeletonize(filled.astype(bool))
    contour_skel[y0:y1, x0:x1] = contour_skel_crop

    A, B = sam_skeleton.astype(bool), contour_skel.astype(bool)
    intersection = int(np.logical_and(A, B).sum())
    a_sum, b_sum = int(A.sum()), int(B.sum())
    dice    = (2*intersection)/(a_sum+b_sum) if (a_sum+b_sum) > 0 else 0.0
    iou_val = intersection/int(np.logical_or(A,B).sum()) if np.logical_or(A,B).any() else 0.0
    return contour_skel, intersection, dice, iou_val


def overlay_masks(frame_bgr, track_dict, track_skeletons=None, alpha=0.5):
    overlay = frame_bgr.copy()
    for tid, mask in track_dict.items():
        color_bgr = WORM_COLORS[tid % len(WORM_COLORS)][::-1]
        overlay[mask == 1] = color_bgr
        ys, xs = np.where(mask == 1)
        if len(xs) == 0: continue
        cx, cy = int(xs.mean()), int(ys.mean())
        if track_skeletons and tid in track_skeletons:
            for sy, sx in zip(*np.where(track_skeletons[tid])):
                cv2.circle(overlay, (int(sx), int(sy)), 1, (255,255,255), -1)
        cv2.putText(overlay, f'W{tid}', (cx-10, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2, cv2.LINE_AA)
    # Fixed: Added the missing 'gamma' argument (usually 0 for simple blending)
    return cv2.addWeighted(frame_bgr, 1-alpha, overlay, alpha, 0)


print('✓ All helper functions defined (with speed optimizations).')

✓ All helper functions defined (with speed optimizations).


In [ ]:
# Cell 6: Core per-video processing function

def process_video(video_path, out_dir):
    """
    Run the full SAM segmentation pipeline on a single video.
    Outputs are saved to out_dir:
      - <name>.segmented_multi.avi   (annotated video)
      - <name>.motion_data.csv       (per-frame worm measurements)

    Returns True on success, False if no worms detected in the first frame.
    """
    os.makedirs(out_dir, exist_ok=True)
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    out_video_path = os.path.join(out_dir, base_name + '.segmented_multi.avi')
    out_csv_path   = os.path.join(out_dir, base_name + '.motion_data.csv')

    # Open video & read metadata
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'  ERROR: cannot open {video_path}')
        return False
    fps          = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    print(f'  Video: {width}x{height} @ {fps} fps | {total_frames} frames')

    # Build background model
    print('  Building background model …')
    background = build_background(video_path)

    #  Detect & segment first frame
    cap = cv2.VideoCapture(video_path)
    ret, first_bgr = cap.read()
    cap.release()
    if not ret:
        print('  ERROR: cannot read first frame.')
        return False

    first_rgb  = cv2.cvtColor(first_bgr, cv2.COLOR_BGR2RGB)
    first_gray = cv2.cvtColor(first_bgr, cv2.COLOR_BGR2GRAY)
    init_boxes, _ = detect_worm_boxes(first_gray, background)

    if len(init_boxes) == 0:
        print('  WARNING: no worms detected in first frame — skipping this video.')
        print('           Try adjusting MIN_ASPECT_RATIO / MIN_SOLIDITY / BG_THRESH.')
        return False

    if len(init_boxes) > 100:
        print(f'  WARNING: {len(init_boxes)} candidate worms detected. Skipping.')
        return False

    print(f'  Detected {len(init_boxes)} worm(s) in first frame.')

    # SPEED FIX: one batched SAM call for all boxes instead of a per-box loop
    init_masks = segment_boxes_batched(predictor, first_rgb, init_boxes)

    next_id = [len(init_masks)]
    tracks  = {i: m for i, m in enumerate(init_masks)}
    track_skeletons = {}
    for tid, mask in tracks.items():
        skel, length_px, endpoints, curvature = skeletonize_mask(mask)
        track_skeletons[tid] = skel

    print(f'{len(tracks)} tracks and {len(track_skeletons)} track skeletons in first frame.')

    # ── Set up output writer & CSV ────────────────────────────────────────────
    fourcc     = cv2.VideoWriter_fourcc(*'XVID')
    out_writer = cv2.VideoWriter(out_video_path, fourcc, fps, (width, height))
    csv_file   = open(out_csv_path, 'w', newline='')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow([
        'frame', 'worm_id',
        'centroid_x', 'centroid_y',
        'speed_px_per_frame', 'heading_deg',
        'body_length_px', 'curvature_rad_per_step',
        'n_endpoints',
        'contour_skel_px', 'skel_overlap_px', 'skel_dice', 'skel_iou',
    ])

    # Write the (already-processed) first frame
    blended = overlay_masks(first_bgr, tracks, track_skeletons)
    out_writer.write(blended)

    # also write a still image of first frame - in case writer doesn't complete
    out_frame0_result_path = os.path.join(out_dir, base_name + '.frame0_result.png')
    cv2.imwrite(out_frame0_result_path,blended)

    # ── Process all remaining frames ──────────────────────────────────────────
    cap = cv2.VideoCapture(video_path)
    cap.read()  # skip first frame (already done)
    prev_centroids = {}
    frame_idx = 1

    import time
    t_start = time.time()

    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break

        frame_rgb  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frame_gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

        # Bounding boxes from existing tracks
        track_boxes, track_box_ids = [], []
        for tid, mask in tracks.items():
            ys, xs = np.where(mask == 1)
            if len(xs) == 0: continue
            x1 = max(0, xs.min()-BOX_PADDING); y1 = max(0, ys.min()-BOX_PADDING)
            x2 = min(width-1, xs.max()+BOX_PADDING); y2 = min(height-1, ys.max()+BOX_PADDING)
            track_boxes.append(np.array([x1, y1, x2, y2]))
            track_box_ids.append(tid)

        # Connected-component detection
        det_boxes, _ = detect_worm_boxes(frame_gray, background)

        # Merge boxes (deduplicate with tracked)
        all_boxes = list(track_boxes)
        for db in det_boxes:
            overlap = any(
                (lambda ix1, iy1, ix2, iy2: (ix2>ix1 and iy2>iy1 and
                 (ix2-ix1)*(iy2-iy1)/((db[2]-db[0])*(db[3]-db[1])+1e-6) > 0.5))(
                     max(db[0],tb[0]), max(db[1],tb[1]),
                     min(db[2],tb[2]), min(db[3],tb[3]))
                for tb in track_boxes
            )
            if not overlap:
                all_boxes.append(db)

        if not all_boxes:
            out_writer.write(frame_bgr)
            frame_idx += 1
            continue

        # SPEED FIX: one batched SAM call for all boxes in this frame
        new_masks = segment_boxes_batched(predictor, frame_rgb, all_boxes)

        # Match to tracks
        tracks = match_masks_to_tracks(tracks, new_masks,
                                        iou_thresh=TRACKING_IOU_THRESH,
                                        next_id_ref=next_id)
        # Skeletonize + motion + CSV
        track_skeletons = {}
        for tid, mask in tracks.items():
            skel, length_px, endpoints, curvature = skeletonize_mask(mask)
            track_skeletons[tid] = skel
            contour_skel, overlap_px, dice, iou_val = contour_skeleton_and_overlap(mask, skel)
            contour_skel_px = int(contour_skel.sum())
            ys, xs = np.where(mask == 1); # Fix: Add a check for empty masks
            if len(xs) == 0: continue # Fix: Skip if no pixels in mask
            cx, cy = float(xs.mean()), float(ys.mean())
            if tid in prev_centroids:
                pcx, pcy = prev_centroids[tid]
                speed   = float(np.sqrt((cx-pcx)**2 + (cy-pcy)**2))
                heading = float(np.degrees(np.arctan2(cy-pcy, cx-pcx)))
            else:
                speed, heading = 0.0, 0.0
            prev_centroids[tid] = (cx, cy)
            csv_writer.writerow([
                frame_idx, tid,
                round(cx, 2), round(cy, 2),
                round(speed, 3), round(heading, 2),
                length_px, round(curvature, 4),
                len(endpoints),
                contour_skel_px, overlap_px,
                round(dice, 4), round(iou_val, 4),
            ])

        # Write annotated frame
        out_writer.write(overlay_masks(frame_bgr, tracks, track_skeletons))
        frame_idx += 1

        # SPEED FIX: one lightweight progress line every 25 frames instead of
        # 6-7 separate flushed print statements EVERY frame (the constant
        # output redraw in Colab has real overhead at hundreds of frames).
        if frame_idx % 25 == 0 or frame_idx == total_frames:
            elapsed = time.time() - t_start
            rate = frame_idx / elapsed if elapsed > 0 else 0
            eta_min = (total_frames - frame_idx) / rate / 60 if rate > 0 else float('nan')
            print(f'  Frame {frame_idx}/{total_frames} | active tracks: {len(tracks)} '
                  f'| {rate:.1f} fps | ETA {eta_min:.1f} min', flush=True)

    print('\n')
    cap.release()
    out_writer.release()
    csv_file.close()
    print(f'  ✓ Saved: {os.path.basename(out_video_path)}')
    print(f'  ✓ Saved: {os.path.basename(out_csv_path)}')
    return True


print('✓ process_video() function defined.')

✓ process_video() function defined.


In [ ]:
# ── Cell: Run the pipeline on your uploaded video ─────────────────────────

out_dir = OUTPUT_ROOT

print(f'Processing: {video_name}')
print('─' * 60)

try:
    success = process_video(video_path, out_dir)
    if success:
        print('\n✓ Done.')
    else:
        print('\n⚠ No worms detected / video skipped — see messages above.')
except Exception as e:
    print(f'\n✗ ERROR: {e}')
    raise

Processing: 20mM caffeine 3minutes.MOV
────────────────────────────────────────────────────────────
  Video: 1080x1920 @ 29 fps | 919 frames
  Building background model …
  Detected 24 worm(s) in first frame.
24 tracks and 24 track skeletons in first frame.
  Frame 25/919 | active tracks: 35 | 0.2 fps | ETA 80.5 min
  Frame 50/919 | active tracks: 43 | 0.1 fps | ETA 114.5 min
  Frame 75/919 | active tracks: 46 | 0.1 fps | ETA 135.4 min
  Frame 100/919 | active tracks: 52 | 0.1 fps | ETA 153.4 min
  Frame 125/919 | active tracks: 54 | 0.1 fps | ETA 167.3 min
  Frame 150/919 | active tracks: 57 | 0.1 fps | ETA 176.0 min
  Frame 175/919 | active tracks: 59 | 0.1 fps | ETA 181.1 min
  Frame 200/919 | active tracks: 62 | 0.1 fps | ETA 185.5 min
  Frame 225/919 | active tracks: 64 | 0.1 fps | ETA 189.8 min
  Frame 250/919 | active tracks: 64 | 0.1 fps | ETA 191.0 min
  Frame 275/919 | active tracks: 64 | 0.1 fps | ETA 189.9 min
  Frame 300/919 | active tracks: 64 | 0.1 fps | ETA 187.4 min
  

In [ ]:
# ── Cell: Verify output files ─────────────────────────────────────────────

print(f'Output folder: {OUTPUT_ROOT}\n')
for root, dirs, files_ in os.walk(OUTPUT_ROOT):
    dirs.sort(); files_.sort()
    level = root.replace(OUTPUT_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files_:
        size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
        print(f'{indent}  {f}  ({size_mb:.1f} MB)')


Output folder: /content/output

output/
  20mM caffeine 3minutes.frame0_result.png  (3.0 MB)
  20mM caffeine 3minutes.motion_data.csv  (4.5 MB)
  20mM caffeine 3minutes.segmented_multi.avi  (65.3 MB)


In [ ]:
# ── Cell 9 (Optional): Quick summary plots for all CSVs ───────────────────────
# Loads every motion_data.csv and plots per-condition speed distributions.

import pandas as pd
import os
from scipy.stats import gaussian_kde # Added for kdeplot with matplotlib

csv_files = []
for root, _, files in os.walk(OUTPUT_ROOT):
    for f in files:
        if f.endswith('.motion_data.csv'):
            condition = os.path.basename(root)
            csv_files.append((condition, os.path.join(root, f)))

if not csv_files:
    print('No CSV files found yet — run the batch pipeline first.')
else:
    all_data = []
    for condition, csv_path in csv_files:
        df = pd.read_csv(csv_path)
        df['condition'] = condition
        df['source_file'] = os.path.basename(csv_path)
        all_data.append(df)
    combined = pd.concat(all_data, ignore_index=True)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    conditions = combined['condition'].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(conditions)))

    for ax, metric, label in zip(
        axes,
        ['speed_px_per_frame', 'body_length_px', 'curvature_rad_per_step'],
        ['Speed (px/frame)', 'Body length (px)', 'Curvature (rad/step)']
    ):
        for cond, color in zip(conditions, colors):
            sub = combined[combined['condition'] == cond][metric].dropna()
            if not sub.empty:
                # Replaced sns.kdeplot with matplotlib equivalent
                kde = gaussian_kde(sub)
                x_range = np.linspace(sub.min(), sub.max(), 500)
                ax.plot(x_range, kde(x_range), label=cond, color=color, alpha=0.7, linewidth=2)
        ax.set_xlabel(label)
        ax.set_ylabel('Density')
        ax.set_title(label)
        ax.legend(fontsize=7)

    plt.suptitle('C. elegans motion summary — all conditions', y=1.02, fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_ROOT, 'summary_plots.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Summary plot saved to {OUTPUT_ROOT}/summary_plots.png')

    print('\nPer-condition mean statistics:')
    print(combined.groupby('condition')[[
        'speed_px_per_frame', 'body_length_px', 'curvature_rad_per_step'
    ]].mean().round(3))

NameError: name 'os' is not defined

In [ ]:
# ── Cell: Download your results ───────────────────────────────────────────
# Downloads the annotated video, the motion-data CSV, and the summary plot
# straight to your computer (each pops as a separate browser download).
# They still need to be written to OUTPUT_ROOT first — the processing/plot
# cells above create them there — but this cell hands you your own copies
# so you don't need to go dig them out of that folder.

from google.colab import files as colab_files

base_name = os.path.splitext(video_name)[0]
to_download = [
    os.path.join(OUTPUT_ROOT, base_name + '.segmented_multi.avi'),
    os.path.join(OUTPUT_ROOT, base_name + '.motion_data.csv'),
    os.path.join(OUTPUT_ROOT, 'summary_plots.png'),
]

for path in to_download:
    if os.path.exists(path):
        colab_files.download(path)
    else:
        print(f'Skipping (not found): {path}')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>